# Treinamento Python — Nível 8 — Python Avançado

> **Data Engineering Track** | Python do zero ao Data Engineering
> Cada seção termina com um **exercício de fixação** e ao final há um **desafio integrador**.

---
# 📘 Aula 01 Decorators

## NÍVEL 8 — Avançado | Aula 1: Decorators

Decorator = função que envolve outra função para adicionar comportamento
Sem modificar o código original da função decorada.

In [ ]:
import time
import functools
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

## 1. Entendendo a base — closures

In [ ]:
def saudacao_factory(saudacao):
    def saudar(nome):
        return f"{saudacao}, {nome}!"
    return saudar

oi = saudacao_factory("Olá")
hey = saudacao_factory("Hey")
print(oi("Ana"))    # Olá, Ana!
print(hey("Bruno")) # Hey, Bruno!

## 2. Decorator básico

In [ ]:
def meu_decorator(funcao):
    @functools.wraps(funcao)   # preserva o nome e docstring da função original
    def wrapper(*args, **kwargs):
        print(f"Antes de chamar {funcao.__name__}")
        resultado = funcao(*args, **kwargs)
        print(f"Depois de chamar {funcao.__name__}")
        return resultado
    return wrapper

@meu_decorator
def dizer_oi(nome):
    print(f"Oi, {nome}!")

dizer_oi("Ana")

# O @meu_decorator é equivalente a: dizer_oi = meu_decorator(dizer_oi)

## 3. Decorators práticos para Data Engineering

Medir tempo de execução

In [ ]:
def medir_tempo(funcao):
    @functools.wraps(funcao)
    def wrapper(*args, **kwargs):
        inicio = time.perf_counter()
        resultado = funcao(*args, **kwargs)
        fim = time.perf_counter()
        logger.info(f"{funcao.__name__} executou em {fim - inicio:.4f}s")
        return resultado
    return wrapper

# Retry automático
def retry(tentativas=3, delay=0.1):
    def decorator(funcao):
        @functools.wraps(funcao)
        def wrapper(*args, **kwargs):
            for i in range(tentativas):
                try:
                    return funcao(*args, **kwargs)
                except Exception as e:
                    if i == tentativas - 1:
                        logger.error(f"{funcao.__name__} falhou após {tentativas} tentativas: {e}")
                        raise
                    logger.warning(f"{funcao.__name__} falhou (tentativa {i+1}/{tentativas}): {e}")
                    time.sleep(delay)
        return wrapper
    return decorator

# Cache simples
def cache(funcao):
    _cache = {}
    @functools.wraps(funcao)
    def wrapper(*args):
        if args not in _cache:
            _cache[args] = funcao(*args)
            logger.debug(f"Cache MISS: {funcao.__name__}{args}")
        else:
            logger.debug(f"Cache HIT: {funcao.__name__}{args}")
        return _cache[args]
    return wrapper

# Validar argumentos
def validar_positivo(funcao):
    @functools.wraps(funcao)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, (int, float)) and arg < 0:
                raise ValueError(f"Argumento negativo não permitido em {funcao.__name__}: {arg}")
        return funcao(*args, **kwargs)
    return wrapper


# Aplicando os decorators
@medir_tempo
@cache
def calcular_fatorial(n):
    if n <= 1:
        return 1
    return n * calcular_fatorial(n - 1)

@medir_tempo
@retry(tentativas=3, delay=0.05)
def buscar_dados(fonte):
    import random
    if random.random() < 0.5:
        raise ConnectionError(f"Falha ao conectar em {fonte}")
    return {"registros": 1500}

@validar_positivo
def calcular_juros(valor, taxa):
    return valor * (1 + taxa)

# Testes
print(calcular_fatorial(10))
print(calcular_fatorial(10))   # do cache

try:
    resultado = buscar_dados("db.empresa.com")
    print(resultado)
except ConnectionError as e:
    print(f"Falhou: {e}")

print(calcular_juros(1000, 0.1))

try:
    calcular_juros(-500, 0.1)
except ValueError as e:
    print(f"Erro: {e}")

## 4. Decorator de classe

In [ ]:
def singleton(cls):
    """Garante que só existe uma instância da classe."""
    _instancias = {}
    @functools.wraps(cls)
    def get_instancia(*args, **kwargs):
        if cls not in _instancias:
            _instancias[cls] = cls(*args, **kwargs)
        return _instancias[cls]
    return get_instancia

@singleton
class ConexaoDB:
    def __init__(self, host):
        self.host = host
        print(f"Nova conexão criada para {host}")

c1 = ConexaoDB("db.empresa.com")
c2 = ConexaoDB("db.empresa.com")
print(c1 is c2)   # True — mesma instância

## EXERCÍCIO DE FIXAÇÃO 8.1

Crie um decorator @log_chamada que:
  - Registra com logging os argumentos da chamada
  - Registra o valor de retorno
  - Registra se ocorreu exceção
Aplique em uma função de processamento de dados.

In [ ]:
# Escreva seu código aqui


---
# 📘 Aula 02 Generators E Context Managers

## NÍVEL 8 — Avançado | Aula 2: Generators e Context Managers

In [ ]:
import time
from contextlib import contextmanager

## 1. Generators — processamento sob demanda

Generator = função que usa yield em vez de return
Não carrega tudo na memória — gera um valor por vez
ESSENCIAL para processar grandes volumes de dados

Função normal: cria toda a lista na memória

In [ ]:
def quadrados_lista(n):
    return [i ** 2 for i in range(n)]

# Generator: gera um valor de cada vez
def quadrados_generator(n):
    for i in range(n):
        yield i ** 2

# Comparação de memória
import sys
lista = quadrados_lista(10000)
gen   = quadrados_generator(10000)

print(f"Lista: {sys.getsizeof(lista):,} bytes")   # ~80KB
print(f"Generator: {sys.getsizeof(gen)} bytes")    # ~200 bytes!

# Iterando sobre o generator
gen = quadrados_generator(5)
for valor in gen:
    print(valor)   # 0, 1, 4, 9, 16

# next() manualmente
gen = quadrados_generator(3)
print(next(gen))   # 0
print(next(gen))   # 1
print(next(gen))   # 4
# next(gen)        → StopIteration

## 2. Generator para processamento de arquivos grandes

In [ ]:
def ler_csv_em_lotes(caminho, tamanho_lote=2):
    """Lê um CSV linha por linha, cedendo lotes."""
    import csv
    from pathlib import Path

    if not Path(caminho).exists():
        # Simular arquivo para o exemplo
        dados_simulados = [
            {"id": str(i), "valor": str(i * 10.5)}
            for i in range(1, 11)
        ]
        lote = []
        for linha in dados_simulados:
            lote.append(linha)
            if len(lote) == tamanho_lote:
                yield lote
                lote = []
        if lote:
            yield lote
        return

    with open(caminho, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        lote = []
        for linha in reader:
            lote.append(linha)
            if len(lote) == tamanho_lote:
                yield lote
                lote = []
        if lote:
            yield lote

for i, lote in enumerate(ler_csv_em_lotes("arquivo_grande.csv", tamanho_lote=3)):
    print(f"Lote {i+1}: {len(lote)} registros — {lote}")

## 3. Generator com send() — comunicação bidirecional

In [ ]:
def acumulador():
    """Generator que acumula valores enviados via send()."""
    total = 0
    while True:
        valor = yield total
        if valor is None:
            break
        total += valor

acc = acumulador()
next(acc)            # inicializa o generator
print(acc.send(100))  # 100
print(acc.send(250))  # 350
print(acc.send(50))   # 400

## 4. Context Managers — gerenciamento de recursos

Garante que recursos sejam abertos e fechados corretamente

O with que você já usou:
with open("arquivo.txt") as f:  ← isso é um context manager

Criando com classe

In [ ]:
class TemporizadorContexto:
    def __init__(self, nome):
        self.nome = nome

    def __enter__(self):
        self.inicio = time.perf_counter()
        print(f"[{self.nome}] Iniciando...")
        return self

    def __exit__(self, tipo_exc, valor_exc, traceback):
        duracao = time.perf_counter() - self.inicio
        print(f"[{self.nome}] Concluído em {duracao:.4f}s")
        if tipo_exc:
            print(f"[{self.nome}] Exceção capturada: {tipo_exc.__name__}: {valor_exc}")
        return False   # False = não suprime a exceção

with TemporizadorContexto("Processamento"):
    time.sleep(0.1)
    total = sum(range(1_000_000))
    print(f"Soma: {total}")

# Criando com @contextmanager (mais simples)
@contextmanager
def temporizador(nome):
    inicio = time.perf_counter()
    print(f"[{nome}] Iniciando...")
    try:
        yield
    finally:
        duracao = time.perf_counter() - inicio
        print(f"[{nome}] Concluído em {duracao:.4f}s")

with temporizador("Cálculo intenso"):
    resultado = [i**2 for i in range(500_000)]

# Context manager para transação simulada
@contextmanager
def transacao(nome):
    print(f"[TX] Iniciando transação: {nome}")
    try:
        yield
        print(f"[TX] COMMIT: {nome}")
    except Exception as e:
        print(f"[TX] ROLLBACK: {nome} — {e}")
        raise

with transacao("Inserir pedidos"):
    print("  Inserindo 500 pedidos...")
    print("  Atualizando estoque...")

## EXERCÍCIO DE FIXAÇÃO 8.2

1. Crie um generator infinito que gera IDs sequenciais:
   "P0001", "P0002", "P0003", ...
2. Crie um context manager @gerenciar_conexao(host) que:
   - Imprime "Abrindo conexão em {host}"
   - Cede um dict {"host": host, "status": "ativo"}
   - Imprime "Fechando conexão em {host}" no finally

In [ ]:
# Escreva seu código aqui


---
# 📘 Aula 03 Type Hints

## NÍVEL 8 — Avançado | Aula 3: Type Hints

Type hints = anotações de tipo que documentam o código
Python NÃO os valida em runtime, mas IDEs, mypy e linters usam
Em projetos profissionais, type hints são padrão

In [ ]:
from typing import Optional, Union, List, Dict, Tuple, Any, Callable
from typing import TypeVar, Generic
from dataclasses import dataclass, field

## 1. Anotações básicas

In [ ]:
def saudar(nome: str) -> str:
    return f"Olá, {nome}!"

def calcular_media(valores: list[float]) -> float:
    return sum(valores) / len(valores)

def processar(dados: list[dict], ativo: bool = True) -> list[dict]:
    return [d for d in dados if d.get("ativo") == ativo]

# Variáveis também podem ter anotações
nome: str = "Ana"
salario: float = 9500.0
skills: list[str] = ["Python", "SQL", "Spark"]

## 2. Optional — pode ser None

In [ ]:
def buscar_usuario(id: int) -> Optional[dict]:
    """Retorna dict se encontrado, None se não existir."""
    banco = {1: {"nome": "Ana"}, 2: {"nome": "Bruno"}}
    return banco.get(id)

usuario = buscar_usuario(1)
if usuario is not None:
    print(usuario["nome"])

## 3. Union — pode ser um tipo OU outro

In [ ]:
def formatar_valor(valor: Union[int, float]) -> str:
    return f"R${valor:.2f}"

# Python 3.10+: pode usar int | float em vez de Union[int, float]
def formatar_v2(valor: int | float) -> str:
    return f"R${valor:.2f}"

## 4. Dict, List, Tuple com tipos internos

In [ ]:
def agrupar_por_categoria(
    produtos: list[dict[str, Any]]
) -> dict[str, list[dict[str, Any]]]:
    resultado: dict[str, list] = {}
    for p in produtos:
        cat = p["categoria"]
        resultado.setdefault(cat, []).append(p)
    return resultado

## 5. Callable — tipo de funções

In [ ]:
Transformador = Callable[[list[Any]], list[Any]]

def aplicar_pipeline(dados: list[Any], *etapas: Transformador) -> list[Any]:
    resultado = dados
    for etapa in etapas:
        resultado = etapa(resultado)
    return resultado

def remover_nulos(dados: list[Any]) -> list[Any]:
    return [d for d in dados if d is not None]

def para_maiusculas(dados: list[str]) -> list[str]:
    return [d.upper() for d in dados]

resultado = aplicar_pipeline(["ana", None, "bruno", None], remover_nulos, para_maiusculas)
print(resultado)

## 6. dataclass — a forma moderna de criar classes de dados

In [ ]:
@dataclass
class Produto:
    nome: str
    preco: float
    estoque: int = 0
    categorias: list[str] = field(default_factory=list)

    def valor_em_estoque(self) -> float:
        return self.preco * self.estoque

    def __post_init__(self):
        if self.preco < 0:
            raise ValueError("Preço não pode ser negativo")

@dataclass(frozen=True)   # imutável — funciona como tupla nomeada
class Coordenadas:
    latitude: float
    longitude: float

p = Produto("Notebook", 3500.0, 10, ["Eletrônico", "Informática"])
print(p)
print(p.valor_em_estoque())

coord = Coordenadas(-23.5, -46.6)
print(coord)

## 7. TypeVar — tipos genéricos

In [ ]:
T = TypeVar("T")

def primeiro_elemento(lista: list[T]) -> Optional[T]:
    return lista[0] if lista else None

print(primeiro_elemento([1, 2, 3]))       # int
print(primeiro_elemento(["a", "b"]))      # str
print(primeiro_elemento([]))              # None

## EXERCÍCIO DE FIXAÇÃO 8.3

Crie um dataclass RegistroVenda com campos tipados:
  - id: str
  - produto: str
  - quantidade: int
  - preco_unitario: float
  - desconto: float = 0.0
Adicione propriedade calculada: valor_total
Adicione método de classe: from_dict(cls, dados: dict)
Crie função processar_vendas que recebe list[RegistroVenda]
e retorna dict com total e media

In [ ]:
# Escreva seu código aqui


---
# 🏆 Desafio Nivel 8

## NÍVEL 8 — DESAFIO FINAL | Framework de Pipeline com Decorators, Generators e Type Hints

CONTEXTO:
Você vai construir um mini-framework de ETL reutilizável
usando tudo do nível 8: decorators, generators, context managers
e type hints para um código profissional e legível.

In [ ]:
# Escreva seu código aqui
